In [0]:
from pyspark.sql.functions import col, row_number, sum as _sum, round
from pyspark.sql.window import Window


catalog = "beverage_analytics"
schema = "delivery"
table = "kpi_top3_groups_by_region"

# --------------------------------------
# 3. Leitura das tabelas
# --------------------------------------
df_fact_sales = spark.table(f"{catalog}.fact.fact_sales")
df_dim_channel = spark.table(f"{catalog}.dim.dim_channel")
df_dim_region = spark.table(f"{catalog}.dim.dim_region")

# --------------------------------------
# 4. Query KPI
# --------------------------------------
query_4_1 = df_fact_sales.alias("fs") \
    .join(df_dim_channel.alias("ch"), col("fs.trade_chnl_desc") == col("ch.trade_chnl_desc")) \
    .join(df_dim_region.alias("region"), col("fs.btlr_org_lvl_c_desc") == col("region.btlr_org_lvl_c_desc")) \
    .groupBy("region.btlr_org_lvl_c_desc", "ch.trade_group_desc") \
    .agg(round(_sum("fs.volume"), 2).alias("total_sales"))

windowSpec = Window.partitionBy("btlr_org_lvl_c_desc").orderBy(col("total_sales").desc())
top3 = query_4_1.withColumn("rank", row_number().over(windowSpec)).filter("rank <= 3")

# --------------------------------------
# 6. Gravar resultado no SQL Pool
# --------------------------------------
top3.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{catalog}.{schema}.{table}")

print(f"KPI gravado com sucesso em: {catalog}.{schema}.{table}")